# Construcción inicial de Gold desde Silver

Ejecución manual en Azure Databricks.
Lee versiones explícitas de ocho entradas Silver, construye siete
tablas Gold, verifica su persistencia y registra tablas externas.

Los conteos corresponden a la carga aceptada después de M8.
Ejecutar por celdas y revisar cada resultado.
No repetir Run all sobre destinos existentes.
Las escrituras entre tablas no forman una transacción conjunta.

Preparado localmente; ejecución en Azure pendiente.


In [ ]:
from pathlib import Path
import sys

current = Path.cwd().resolve()
repo_root = next(
    (
        path for path in (current, *current.parents)
        if (path / "transformations" / "gold").is_dir()
        and (path / "sources" / "reference").is_dir()
    ),
    None,
)

if repo_root is None:
    raise RuntimeError("No se encontró la raíz del repositorio.")

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from transformations.gold.azure_inputs import read_azure_gold_inputs
from transformations.gold.build import build_gold_tables
from transformations.gold.storage import write_verified_gold_table
from transformations.gold.validation import validate_gold_model

print("PASS: funciones Gold importadas.")


In [ ]:
spark.conf.set("spark.sql.session.timeZone", "UTC")

catalog = "dbw_arlp_dev_diego01"
silver_root = "abfss://silver@starlpdevdiego01.dfs.core.windows.net"
gold_root = "abfss://gold@starlpdevdiego01.dfs.core.windows.net"

silver_entities = {
    "categories": ("techretail/categories", 6),
    "products": ("techretail/products", 61),
    "customers": ("techretail/customers", 500),
    "orders": ("techretail/orders", 2000),
    "order_items": ("techretail/order_items", 5862),
    "dummyjson_products": ("external/dummyjson_products", 38),
    "product_external_mapping": ("reference/product_external_mapping", 6),
    "exchange_rates": ("reference/exchange_rates", 195),
}

print("Entradas Silver configuradas:", len(silver_entities))
print("Destino Gold:", gold_root)


In [ ]:
from delta.tables import DeltaTable

input_specifications = {}

for entity, (relative_path, expected_count) in silver_entities.items():
    path = f"{silver_root}/{relative_path}"
    latest = (
        DeltaTable.forPath(spark, path)
        .history(1)
        .select("version")
        .first()
    )

    if latest is None:
        raise ValueError(f"Sin historial Delta: {entity}")

    input_specifications[entity] = {
        "path": path,
        "version": int(latest["version"]),
        "row_count": expected_count,
    }

inputs, input_records = read_azure_gold_inputs(
    spark, input_specifications
)

for entity, record in input_records.items():
    print(
        f"PASS: {entity}, versión={record['version']}, "
        f"filas={record['row_count']}"
    )


In [ ]:
from datetime import datetime, timezone
from uuid import uuid4

gold_processing_run_id = str(uuid4())
gold_processed_at = datetime.now(timezone.utc)

tables = build_gold_tables(
    inputs,
    gold_processing_run_id,
    gold_processed_at,
)

expected_tables = {
    "dim_product",
    "dim_customer",
    "dim_date",
    "dim_order_status",
    "ref_exchange_rates",
    "product_catalog_mapping",
    "fact_sales",
}

if set(tables) != expected_tables:
    raise ValueError("La construcción no devolvió las siete tablas esperadas.")

print("Ejecución Gold:", gold_processing_run_id)
print("Procesamiento UTC:", gold_processed_at.isoformat())
print("DataFrames Gold preparados:", ", ".join(sorted(tables)))


In [ ]:
import json

gold_targets = {
    "dim_product": f"{gold_root}/sales/dim_product",
    "dim_customer": f"{gold_root}/sales/dim_customer",
    "dim_date": f"{gold_root}/sales/dim_date",
    "dim_order_status": f"{gold_root}/sales/dim_order_status",
    "fact_sales": f"{gold_root}/sales/fact_sales",
    "ref_exchange_rates": f"{gold_root}/reference/ref_exchange_rates",
    "product_catalog_mapping": f"{gold_root}/reference/product_catalog_mapping",
}

if set(gold_targets) != set(tables):
    raise ValueError("Los destinos no coinciden con las tablas construidas.")

report_directory = (
    f"{gold_root}/_executions/run_id={gold_processing_run_id}"
)
report_path = f"{report_directory}/report.json"

report = {
    "scope": "azure_gold_initial",
    "status": "running",
    "gold_processing_run_id": gold_processing_run_id,
    "gold_processed_at": gold_processed_at.isoformat(),
    "inputs": input_records,
    "tables": {},
    "model_validation": "pending",
}


def save_execution_report():
    dbutils.fs.put(
        report_path,
        json.dumps(report, ensure_ascii=False, indent=2) + "\n",
        overwrite=True,
    )


for name, target in gold_targets.items():
    print(f"{name}: {target}")

print("Informe:", report_path)


In [ ]:
if report["status"] != "running" or report["tables"]:
    raise RuntimeError(
        "La escritura ya se inició; revisar el informe antes de continuar."
    )

cached_gold = []
persisted_tables = {}

dbutils.fs.mkdirs(report_directory)
save_execution_report()

try:
    for name, frame in tables.items():
        frame.cache()
        cached_gold.append(frame)

        result = write_verified_gold_table(
            spark, frame, gold_targets[name]
        )
        report["tables"][name] = result
        save_execution_report()

        print(f"PASS: {name}, {result['row_count']} filas verificadas.")

    report["status"] = "tables_verified"
    save_execution_report()

    for name, record in report["tables"].items():
        persisted_tables[name] = (
            spark.read.format("delta")
            .option("versionAsOf", record["version"])
            .load(record["path"])
        )

    report["model_validation"] = validate_gold_model(
        inputs, persisted_tables, report
    )
    report["status"] = "model_validated"
    report["finished_at"] = datetime.now(timezone.utc).isoformat()
    save_execution_report()

    print("PASS: modelo Gold persistido validado.")
    print("Informe:", report_path)

except Exception as error:
    report["status"] = "failed"
    report["error"] = str(error)
    report["finished_at"] = datetime.now(timezone.utc).isoformat()
    save_execution_report()
    raise

finally:
    for frame in cached_gold:
        frame.unpersist()


In [ ]:
if report["status"] != "model_validated":
    raise RuntimeError("Se requiere un modelo persistido validado.")

registered_names = {
    name: (
        f"{catalog}.gold_reference.{name}"
        if name in ("ref_exchange_rates", "product_catalog_mapping")
        else f"{catalog}.gold_sales.{name}"
    )
    for name in gold_targets
}

report["registration"] = {
    "status": "running",
    "tables": {},
}

try:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.gold_sales")
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.gold_reference")

    for full_name in registered_names.values():
        if spark.catalog.tableExists(full_name):
            raise ValueError(
                f"La tabla ya está registrada; revisar su destino: {full_name}"
            )

    save_execution_report()

    for name, full_name in registered_names.items():
        target = gold_targets[name]

        spark.sql(
            f"CREATE TABLE {full_name} "
            f"USING DELTA LOCATION '{target}'"
        )

        registered = spark.table(full_name)
        stored = persisted_tables[name]

        if [
            (field.name, field.dataType) for field in registered.schema
        ] != [
            (field.name, field.dataType) for field in stored.schema
        ]:
            raise ValueError(f"Tipos registrados diferentes: {full_name}")

        if (
            registered.exceptAll(stored).limit(1).count()
            or stored.exceptAll(registered).limit(1).count()
        ):
            raise ValueError(f"Valores registrados diferentes: {full_name}")

        report["registration"]["tables"][name] = full_name
        save_execution_report()
        print("PASS: registro y consulta verificados:", full_name)

    report["registration"]["status"] = "passed"
    report["status"] = "completed"
    report["finished_at"] = datetime.now(timezone.utc).isoformat()
    save_execution_report()

except Exception as error:
    report["registration"]["status"] = "failed"
    report["status"] = "failed"
    report["error"] = str(error)
    report["finished_at"] = datetime.now(timezone.utc).isoformat()
    save_execution_report()
    raise
